In [ ]:
import numpy as np
import pandas as pd
import nba.db as db
import nba.visualization as vis
import nba.eng as eng
import nba.stats as stats
from nba.data_loader import download_yearly_stats
import matplotlib.pyplot as plt
import seaborn as sns
import importlib


# For development!
importlib.reload(db)
importlib.reload(vis)
importlib.reload(eng)
importlib.reload(stats)

In [ ]:
# Connect to db
con = db.connect()

### Question 1: How does probability of winning change with lead at halftime?
Over the last 20 years!

In [ ]:
df = db.query_line_score(con, 2003, 2023)
df.head()

In [ ]:
df = eng.add_halftime_features(df)
df.head()

Take a quick look at distribution of half-time leads.

In [ ]:
vis.home_lead_hist(df)

In [ ]:
home_probs = eng.compute_halftime_win_probabilities(df, False) 
display(home_probs.head())

In [ ]:
sns.lineplot(home_probs)

New question: is this distribution symmetric, i.e. if we model win probabilities as a function of lead at the half, does it matter if the home or away team is leading?

We can model this as a logistic regression, where we assume
$$ \log (\frac{\mathbb{P}(Y_i = 1)}{\mathbb{P}(Y_i = 0)}) = x_i^\top \beta^0 $$
where $Y_i = 1$ if the home team wins the $i^{th}$ game and $0$ otherwise, and $x_i$ is a vector of covariates for the $i^{th}$ game.

We want to test whether:
1. There is an intercept for the regression, i.e. a tie has equal probability of home/away team winning.
2. Is the distribution symmetric, or is there a different slope for home and away leads?

We can formalize this with the following model:
$$ \tilde Y_i = \beta_0 + \beta_1 x_i + \beta_2 |x_i|$$
where $x_i$ is the home teams lead at half, and $\tilde Y_i$ is the log-odds of the home team winning. The absolute value term gives different coefficients for home/away leads. Observe that if $x_i > 0$, we have $\beta_0 + (\beta_1 + \beta_2)x_i$ and if $x_i < 0$, we have $\beta_0 + (\beta_1 - \beta_2)x_i$. We test the null hypothesis $ \beta_0 = \beta_2 = 0$, for the
completely symmetric case.

In [ ]:
model, wald_results = stats.run_halftime_log_reg(df)
print(model.summary())
print(wald_results.summary())

#### It looks like it's pretty symmetric! We do not reject the null.

### Question 2. Do NBA players have long arms relative to their height?

In [ ]:
df = db.query_combine_stats(con)
df.head()

In [ ]:
df = eng.compute_wingspan_ratio(df)
df.head()

In [ ]:
vis.graph_wingspan_ratio(df, None)

The red line above is the mean wingspan/height ratio, around 1.06. The average in the population is close to 1! This shows that NBA players, on average, have much longer arms relative to their height than your typical person.

In [ ]:
vis.graph_position_boxplot(df, "wingspan/height")

In [ ]:
vis.graph_position_boxplot(df, "height_wo_shoes")

### Question 3: Are any combine stats highly correlated with career games played in the NBA?

In [ ]:
# Download stats for players from 2000-2022
years = list(range(2000, 2023, 1))
download_yearly_stats(years, 2)

In [ ]:
df = pd.read_csv("data/season_player_stats.csv")
df = eng.filter_complete_careers(df, 2000, 2022)
df.head()

In [ ]:
df = eng.add_gp_to_combine(df)
df.sort_values(by="gp", ascending=False).head()

In [ ]:
sns.heatmap(df.select_dtypes(include="number").drop("player_id", axis=1).corr(), cmap="rocket_r")

In summary, not really!

### Question 4: How does players productivity change over time?

In [ ]:
df = pd.read_csv("data/season_player_stats.csv")
df.head()

In [ ]:
df = eng.add_player_tenure_and_ppg(df)
df.head()

In [ ]:
vis.graph_mean_ppg_over_career(df)

Unsurprisingly, players seem to peak towards the middle of their careers.